<img src="https://raw.githubusercontent.com/IDEALLab/EngiOpt/codex/dcc26-workshop-notebooks/workshops/dcc26/assets/engibench_logo.png" width="560"/>

# Notebook 00 — Framing your design problem

> **Colab users:** click **File ➜ Save a copy in Drive** before editing so your changes persist.

## Where we are in the workshop

You just heard a talk arguing that **engineering design needs better benchmarks** before we can seriously compare ML methods. In this notebook we explore the tools contained in [EngiBench](https://github.com/IDEALLab/EngiBench) which were designed to address these needs. 

**You do not need any ML background for this notebook.** Just Python.


## A concrete problem to anchor the discussion

<img src="https://raw.githubusercontent.com/IDEALLab/EngiOpt/codex/dcc26-workshop-notebooks/workshops/dcc26/assets/engibench_problems.png" width="700"/>

Imagine a colleague walks into your office and says:

> *"I need a truss structure that holds a known load at a known point, using no more than 40% of the available material. Can ML help me design it?"*

But this leaves a lot of unanswered questions that must be explicit for proper benchmarking. Before proceeding it would be nice to know:

1. **What is the space of allowable designs?** (Images/Meshes? Resolution?)
2. **What conditions is this deisgn operating under?** (Where are the forces? Where are the supports?)
3. **What does "better" actually mean?** (Lowest weight? Smallest deflection?)
4. **Do I have reference designs to learn from or compare against?**
5. **How can I visualize a solution?** (Visuals compliment numbers)
6. **How do I know my design works?** (Can I simulate it?)
8. **How can I compare the ML-design to classical design?** (Optimality gap?)

Without knowing the answer to these questions, we cannot properly design or evaluate an ML model.

## Install dependencies (Colab / fresh env only)

Skip this if your local environment already has `engibench` installed.

In [ ]:
import subprocess, sys

IN_COLAB = "google.colab" in sys.modules
FORCE_INSTALL = False  # flip to True to force install locally

if IN_COLAB or FORCE_INSTALL:
    def _pip(pkgs): subprocess.check_call([sys.executable, "-m", "pip", "install", *pkgs])
    _pip(["engibench[all]", "matplotlib", "ipywidgets"])
    _pip(["git+https://github.com/IDEALLab/EngiOpt.git@codex/dcc26-workshop-notebooks#egg=engiopt"])
    print("Install complete.")
else:
    print("Using current environment. Set FORCE_INSTALL=True to install here.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from engibench.utils.all_problems import BUILTIN_PROBLEMS

SEED = 7
np.random.seed(SEED)

---
## *Defining the EngiBench Problem*

Every existing problem in EngiBench has a string ID. 

For this workshop we will start with the `beams2d` problem which minimizes the compliance (flex) of a 2D beam under a prescribed load, subject to a material-budget constraint.

Below we pull the builtin `beams2d` problem class. This line can be adapted to pull other EngiBench problem classes like `airfoil`, `heatconduction2d`, `photonics2d`...


In [ ]:
problem = BUILTIN_PROBLEMS["beams2d"](seed=SEED)
print(f"Loaded: {type(problem).__name__}")

---
## 1 — *What is the space of allowable designs?*

Before we design any ML models, we have to know the **shape of the thing you're producing**. An ML model that outputs a 3D mesh is not interchangeable with one that outputs a 2D image.

In EngiBench this is defined in a problem's **design space**. The design space tells us the shape, resolution, and datatype of designs.


In [ ]:
print("Design space:", problem.design_space)
print("Design shape:", problem.design_space.shape)

**Reading that output.** A design here is a **50 × 100 array of floats in [0, 1]**. You can picture it as a grayscale image, where each pixel is the *density of material* at that location: 0 means empty, 1 means fully solid.


---
## 2 — *What conditions is this deisgn operating under?*

A good 2D beam design is only *good* under a specific operating scenario. The *best* beam when the force is placed halfway between support is not the *best* beam when the load is applied directly over a support.

The scenario you evaluate under is called the **operating condition** (or just *condition*).

Let's see which conditions `beams2d` needs.


In [ ]:
print("Condition keys:", problem.conditions_keys)

For `beams2d` these are:

- `volfrac` — the fraction of the design area allowed to be solid material (your budget).
- `forcedist` — where the load is applied along the structure.
- `rmin` — the filter radius used by the solver; effectively the smallest feature size the result is allowed to resolve.
- `overhang_constraint` — whether the design must be manufacturable by additive processes without support overhangs.

Every design in this benchmark is evaluated **relative to a specific setting of these conditions**. 


---
## 3 — *What does "better" actually mean?*

What actually makes a beam better than another one? 

We must define the problem **objective**. In EngiBench the objective contains both a scalar name and the desired direction (minimize or maximize) that indicates better performance.


In [ ]:
print("Objectives:", problem.objectives)

For `beams2d` the objective is **compliance** (how much the structure deflects under load) and we want to **minimize** it — stiffer is better.


---
## 4 — *Do I have reference designs to learn from or compare against?*

If we want to develop an ML model for this problem we need:

- **Training material** if you're going to fit a model — you need examples of *good* designs under varied conditions.
- **Comparison points** out of sample from the training data to evaluate unbiased performance of the ML model

A benchmark therefore ships with a **dataset** of known-good designs, each paired with the condition it was optimized for.

In [ ]:
dataset = problem.dataset
print(dataset)

Here we see the train(3880 designs)/test(243 designs)/validation(728 designs) splits for the `beams2d` dataset. Each element of the dataset contains the `optimal_design` from traditional topology optimization and the conditions it was optimized under as well as the compliance `c` achieved.

### Visualize the design/condition pairs in the dataset 

Drag the sliders below to filter the dataset by condition ranges. Narrow `volfrac` and the surviving designs get thinner; slide `forcedist` and the load — and the truss pattern solving it — moves.

In [ ]:
from engiopt.workshops.dcc26.notebook_helpers import interactive_condition_explorer

interactive_condition_explorer(dataset, problem)

In [ ]:
# Choose an individual sample design from the dataset
sample_idx = 50
design = np.array(dataset["train"]["optimal_design"][sample_idx])
config = {k: np.asarray(dataset["train"][k][sample_idx]) for k in problem.conditions_keys}

# Display the design and scalar conditions for this sample
print("Design shape:", design.shape)
print("Scalar conditions for this sample:")
for k, v in config.items():
    if np.asarray(v).ndim == 0:
        print(f"  {k} = {float(v):.3f}")

 **Each dataset entry is a (design, scenario) pair**. That's what a benchmark dataset has to be. A pile of designs without their scenarios would be useless: you couldn't say what any of them was solving.


---
## 5 — *What does the design look like?*

Numbers only get you so far. At some point, as an engineer, you want to **see** a candidate — does it look plausible? Is the material spread sensibly? Are supports where you'd expect?

A benchmark should give you a canonical way to *visualize* a design, so that discussion across papers is about the same picture.


In [ ]:
fig = problem.render(design)
plt.title(f"A beams2d reference design (sample {sample_idx})")
plt.show()

That's a topology-optimized beam: red pixels are material, blue pixels are void. 

---
## 6 — *How well does this design actually perform?*

Ok, it looks like a beam. The next question is: how well does it perform, or **what's the objective value?**. 

For that we need a **simulator** — a function that takes `(design, condition)` and returns the physics score --> in this case the compliance `c` of the beam.

In [ ]:
problem.reset(seed=SEED)  # clear cached FEM state from any earlier simulate call
obj_values = problem.simulate(design, config=config)
print(f"Objective(s): {problem.objectives}")
print(f"Values for this design: {obj_values}")

---
## 7 — *How can I compare the ML-design to classical design?*

Before you celebrate a new method, you need to know what it has to beat. For most engineering problems there is already **a classical optimizer** that has been the workhorse for decades. 

Running the classical optimizer from a trivial starting point (like a constant volfrac) yields a *reference* design and a *trajectory* showing how quickly the solver converges. Your ML method has to be compared against both — not just final quality, but how fast the baseline got there.

> **Heads-up:** this cell runs a real FEM optimizer and can take ~30s depending on hardware.

In [ ]:
problem.reset(seed=SEED)  # clear cached FEM state so optimize starts fresh

# A "trivial" starting point depends on the problem. If there is a material-budget
# condition we can fill the field at that level; otherwise fall back to the midpoint
# of the design space (e.g. photonics2d, which has no volume fraction).
if "volfrac" in problem.conditions_keys:
    fill = float(config["volfrac"])
elif "volume" in problem.conditions_keys:
    fill = float(config["volume"])
else:
    low, high = float(problem.design_space.low.min()), float(problem.design_space.high.max())
    fill = 0.5 * (low + high)

starting_point = np.full(problem.design_space.shape, fill)
optimized_design, history = problem.optimize(starting_point, config)

print(f"Optimizer ran for {len(history)} steps")
print(f"Final objective: {history[-1].obj_values}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(design, cmap="gray_r", vmin=0, vmax=1)
axes[0].set_title(f"Dataset design (obj={obj_values[0]:.3f})")
axes[0].axis("off")
axes[1].imshow(optimized_design, cmap="gray_r", vmin=0, vmax=1)
axes[1].set_title(f"Freshly optimized (obj={history[-1].obj_values[0]:.3f})")
axes[1].axis("off")
plt.tight_layout(); plt.show()

Notice the two designs look similar but not identical — classical topology optimizers are non-convex and depend on starting point. That tells you something important: **the "right answer" isn't unique**. Your benchmark has to handle that with diversity-aware metrics, which is exactly what Notebook 02 will do.

---
## 8 — *When is a candidate design actually invalid?*

Here's a failure mode that catches ML researchers by surprise: your model can have low training loss *and still emit garbage*. Pixel-MSE doesn't know about physics. The network will happily output a "design" with negative material density or zero material where the solver needs at least a little.

A benchmark therefore needs an **independent validity test** — a separate function that says yes-or-no *"this candidate obeys the rules"* regardless of how it was produced.

EngiBench calls these **constraints**, and tags each one by *why* it exists:

| Category | Meaning |
|---|---|
| `THEORY` | Comes from **physics**. Values outside this are unphysical (e.g. negative volume fraction). |
| `IMPL`   | Comes from the **solver implementation**. Violating it crashes or destabilises the numerical method. |

Let's first confirm our dataset design is valid under its own scenario (it should be — it came from the solver).

In [ ]:
violations = problem.check_constraints(design=design, config=config)
print(f"Checked {violations.n_constraints} constraints — {len(violations)} violated.")

Now let's feel the other side of it. We'll lie about the volume budget by setting it to something extreme, and ask the benchmark whether the same design still qualifies under that lie.


In [ ]:
bad_config = dict(config)
bad_config["volfrac"] = 0.01  # claim we only have 1% material budget

bad_violations = problem.check_constraints(design=design, config=bad_config)
print(f"Under the bad config: {len(bad_violations)} violation(s).\n")
if bad_violations:
    print(bad_violations)

**What just happened.** The design itself didn't change — only the scenario we claimed it was solving. The benchmark caught the mismatch.

---
## Putting it together

Look back at the engineer's checklist at the top of this notebook. Here's the mapping we just built up:

| Researcher's question | What we need | Where it lives on `problem` |
|---|---|---|
| What is my problem called? | A reproducible identity | `BUILTIN_PROBLEMS["beams2d"]` |
| What am I designing? | A design space | `problem.design_space` |
| Under what scenarios? | Operating conditions | `problem.conditions_keys` |
| What does better mean? | An objective | `problem.objectives` |
| Is there prior work to learn from? | A dataset | `problem.dataset` |
| Can I see a design? | A renderer | `problem.render(design)` |
| How does this design score? | A simulator | `problem.simulate(design, config)` |
| What do I have to beat? | A classical baseline | `problem.optimize(start, config)` |
| When is a design invalid? | A validity check | `problem.check_constraints(design, config)` |

---
## Reflect before moving on

Pick one row in the table above and ask: *what would go wrong in published results if this row were left implicit* (i.e. an ML paper published results without disclosing)?

## Next
In **Notebook 01** we take an EngiBench dataset and train a simple generative model with it. In **Notebook 02** we learn how to use these EngiBench tools to evaluate a generative model.